# Medical Waste Analytics — CPCB 2023
Exploratory analysis of 37 State/UT-level reporting entities. This is a single-year aggregate dataset; it has no monthly observations or waste-colour-category fields.

In [ ]:
from pathlib import Path
import sys, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/'data').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analytics import load_data, numeric_frame, treatment_metrics, build_summary_statistics
df = load_data(ROOT/'data/processed/medical_waste_cleaned.csv')
data = numeric_frame(df)
print(f'{len(df)} reporting entities × {len(df.columns)} columns')
df.head()

## Data quality and coverage
Missing values are not imputed. NIL values were mapped to zero during preprocessing; INP, NA and blank values remain missing.

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame('missing_values'))
assert len(data) == 37 and data.state_ut.is_unique
assert data.bmw_generated_kg_day.notna().all()
assert data.bmw_treated_disposed_kg_day.notna().all()

## Waste generation by reporting entity (kg/day)

In [ ]:
generation = data[['state_ut','bmw_generated_kg_day','bmw_treated_disposed_kg_day']].sort_values('bmw_generated_kg_day', ascending=False)
display(generation.head(10))
ax = generation.head(10).sort_values('bmw_generated_kg_day').plot.barh(x='state_ut', y='bmw_generated_kg_day', legend=False, figsize=(9,6))
ax.set_xlabel('Waste generated (kg/day)'); ax.set_title('Top 10 reporting entities'); plt.tight_layout(); plt.show()

## Generated versus treated/disposed
The ratio is an aggregate comparison, not proof that the difference is untreated waste. Reporting differences may affect the comparison.

In [ ]:
metrics = treatment_metrics(data)
display(pd.Series(metrics).to_frame('value'))
ax = generation.head(10).sort_values('bmw_generated_kg_day').plot.barh(x='state_ut', y=['bmw_generated_kg_day','bmw_treated_disposed_kg_day'], figsize=(10,6))
ax.set_xlabel('kg/day'); plt.tight_layout(); plt.show()

## Treatment route and infrastructure

In [ ]:
route = data[['bmw_treated_captive_kg_day','bmw_treated_cbwtf_kg_day']].sum()
display(route.to_frame('kg_per_day'))
route.plot.pie(autopct='%1.1f%%', figsize=(6,6), ylabel='', title='Reported treatment quantity by route'); plt.show()
display(data[['cbwtfs_operational','cbwtfs_under_construction','total_hcfs','total_beds']].sum().to_frame('total'))

## Distribution and relationships
Correlations are descriptive, not causal. State-level aggregation hides facility-level variation.

In [ ]:
data.bmw_generated_kg_day.plot.hist(bins=12, figsize=(8,4), title='Distribution of waste generation'); plt.xlabel('kg/day'); plt.show()
ax = data.plot.scatter(x='total_hcfs', y='bmw_generated_kg_day', figsize=(7,5)); ax.set_xlabel('Healthcare facilities'); ax.set_ylabel('Waste generated (kg/day)'); plt.show()
display(data[['total_hcfs','total_beds','bmw_generated_kg_day','bmw_treated_disposed_kg_day','cbwtfs_operational']].corr(numeric_only=True))

## Statistics export and validation
No time trend or yellow/red/white/blue category breakdown is possible because those fields are absent.

In [ ]:
summary = build_summary_statistics(data)
out = ROOT/'reports/summary_statistics.csv'; out.parent.mkdir(exist_ok=True)
summary.to_csv(out, index=False)
display(summary.head(20))
assert summary.value.notna().any()
print('Saved', out)